# Proyek Generative AI: Mini DDPM *from Scratch* (PyTorch)
Notebook ini berisi arsitektur inti dari Denoising Diffusion Probabilistic Models (DDPM) yang dibangun murni menggunakan PyTorch (tanpa library Diffusers).

**Tugas Selesai (Arsitektur):**
- Komponen Dasar (Time Embedding, ResBlock, Attention, Sampling ops)
- Arsitektur U-Net (~3.4M Parameters)
- Manajer Difusi (Forward Process, p_losses, Sampling standar, dan Fast Sampling DDIM)

**Tugas Selanjutnya (Tim Training):**
- Setup DataLoader (Fashion-MNIST)
- Optimizer & Training Loop
- Evaluasi & EMA (Opsional)

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Komponen Dasar Jaringan
Berisi modul-modul pembangun untuk U-Net, termasuk embedding waktu sinusoidal, ResBlock dengan arsitektur *pre-activation*, modul Self-Attention yang aman dari isu *contiguous memory*, serta operasi downsample/upsample.

In [2]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        half_dim = dim // 2
        
        emb = math.log(10000) / (half_dim - 1)
        freqs = torch.exp(torch.arange(half_dim, dtype=torch.float32) * -emb)
        self.register_buffer("freqs", freqs, persistent=False)
        
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        t = t.long()
        args = t.float().unsqueeze(1) * self.freqs.unsqueeze(0)
        embeddings = torch.cat([args.sin(), args.cos()], dim=-1)
        
        if self.dim % 2 == 1:
            embeddings = F.pad(embeddings, (0, 1, 0, 0))
            
        return self.mlp(embeddings)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_emb_dim, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(num_groups=groups, num_channels=in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        
        self.norm2 = nn.GroupNorm(num_groups=groups, num_channels=out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x, t_emb):
        h = self.norm1(x)
        h = F.silu(h)
        h = self.conv1(h)
        
        t_hidden = self.time_mlp(t_emb).view(-1, h.shape[1], 1, 1)
        h = h + t_hidden
        
        h = self.norm2(h)
        h = F.silu(h)
        h = self.conv2(h)
        
        return h + self.shortcut(x)


class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4, groups=8):
        super().__init__()
        assert channels % num_heads == 0, "Channels harus habis dibagi num_heads"
        self.norm = nn.GroupNorm(num_groups=groups, num_channels=channels)
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        
        h = h.reshape(B, C, H * W).transpose(1, 2)
        attn_out, _ = self.mha(h, h, h)
        attn_out = attn_out.transpose(1, 2).reshape(B, C, H, W)
        
        return x + attn_out


class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2.0, mode='nearest')
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        return self.conv(self.upsample(x))

## 2. Arsitektur U-Net
Menerima input gambar bernoise $(B, 1, 28, 28)$ dan *timestep* $t$, kemudian memprediksi *noise* yang ditambahkan pada langkah tersebut. Disusun dengan *channel multiplier* [1, 2] dan 1 blok *bottleneck attention*.

In [3]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, time_emb_dim=256):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(dim=64)
        self.time_mlp = nn.Sequential(
            nn.Linear(64, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )
        
        self.init_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        
        self.enc_res1 = ResBlock(base_channels, base_channels, time_emb_dim)
        self.enc_res2 = ResBlock(base_channels, base_channels, time_emb_dim)
        self.down1 = Downsample(base_channels)
        
        self.enc_res3 = ResBlock(base_channels, base_channels * 2, time_emb_dim)
        self.enc_res4 = ResBlock(base_channels * 2, base_channels * 2, time_emb_dim)
        self.down2 = Downsample(base_channels * 2)
        
        self.bot_res1 = ResBlock(base_channels * 2, base_channels * 2, time_emb_dim)
        self.bot_attn = AttentionBlock(base_channels * 2, num_heads=4, groups=8)
        self.bot_res2 = ResBlock(base_channels * 2, base_channels * 2, time_emb_dim)
        
        self.up1 = Upsample(base_channels * 2)
        self.dec_res1 = ResBlock(base_channels * 2 + base_channels * 2, base_channels * 2, time_emb_dim) 
        self.dec_res2 = ResBlock(base_channels * 2 + base_channels * 2, base_channels * 2, time_emb_dim) 
        
        self.up2 = Upsample(base_channels * 2)
        self.dec_res3 = ResBlock(base_channels * 2 + base_channels, base_channels, time_emb_dim)       
        self.dec_res4 = ResBlock(base_channels + base_channels, base_channels, time_emb_dim)           
        
        self.out_norm = nn.GroupNorm(num_groups=8, num_channels=base_channels)
        self.out_conv = nn.Conv2d(base_channels, 1, kernel_size=3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(self.time_emb(t))
        skips = []
        
        x = self.init_conv(x)
        
        x = self.enc_res1(x, t_emb); skips.append(x)
        x = self.enc_res2(x, t_emb); skips.append(x)
        x = self.down1(x)
        
        x = self.enc_res3(x, t_emb); skips.append(x)
        x = self.enc_res4(x, t_emb); skips.append(x)
        x = self.down2(x)
        
        x = self.bot_res1(x, t_emb)
        x = self.bot_attn(x)
        x = self.bot_res2(x, t_emb)
        
        x = self.up1(x)
        x = torch.cat([x, skips.pop()], dim=1)
        x = self.dec_res1(x, t_emb)
        
        x = torch.cat([x, skips.pop()], dim=1)
        x = self.dec_res2(x, t_emb)
        
        x = self.up2(x)
        x = torch.cat([x, skips.pop()], dim=1)
        x = self.dec_res3(x, t_emb)
        
        x = torch.cat([x, skips.pop()], dim=1)
        x = self.dec_res4(x, t_emb)
        
        x = self.out_norm(x)
        x = F.silu(x)
        return self.out_conv(x)

## 3. Manajer Difusi
Mengatur *noise schedule* (linear beta), proses penambahan noise (forward), perhitungan MSE loss (`p_losses`), serta proses *sampling* gambar dari *noise* murni menggunakan DDPM standar (1000 langkah) atau DDIM (*fast sampling*).

In [4]:
class GaussianDiffusion(nn.Module):
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.num_timesteps = num_timesteps
        
        betas = torch.linspace(beta_start, beta_end, num_timesteps, dtype=torch.float32)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cumprod", alphas_cumprod)
        self.register_buffer("sqrt_alphas_cumprod", torch.sqrt(alphas_cumprod))
        self.register_buffer("sqrt_one_minus_alphas_cumprod", torch.sqrt(1.0 - alphas_cumprod))
        
        self.register_buffer("sqrt_recip_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod))
        self.register_buffer("sqrt_recipm1_alphas_cumprod", torch.sqrt(1.0 / alphas_cumprod - 1.0))
        self.register_buffer("posterior_mean_coef1", betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod))
        self.register_buffer("posterior_mean_coef2", (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod))
        self.register_buffer("variance", betas)

    def extract(self, a, t, x_shape):
        batch_size = t.shape[0]
        out = a.gather(-1, t.long())
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar_t = self.extract(self.sqrt_alphas_cumprod, t, x0.shape)
        sqrt_one_minus_alpha_bar_t = self.extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape)
        return sqrt_alpha_bar_t * x0 + sqrt_one_minus_alpha_bar_t * noise

    def p_losses(self, model, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, noise=noise)
        predicted_noise = model(x_t, t)
        return F.mse_loss(predicted_noise, noise)

    @torch.no_grad()
    def p_sample(self, model, x, t, t_index, clip_denoised=True):
        pred_noise = model(x, t)
        
        sqrt_recip_alpha_bar_t = self.extract(self.sqrt_recip_alphas_cumprod, t, x.shape)
        sqrt_recipm1_alpha_bar_t = self.extract(self.sqrt_recipm1_alphas_cumprod, t, x.shape)
        pred_x0 = sqrt_recip_alpha_bar_t * x - sqrt_recipm1_alpha_bar_t * pred_noise
        
        if clip_denoised:
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)
            
        post_coef1_t = self.extract(self.posterior_mean_coef1, t, x.shape)
        post_coef2_t = self.extract(self.posterior_mean_coef2, t, x.shape)
        mean = post_coef1_t * pred_x0 + post_coef2_t * x
        
        if t_index > 0:
            noise = torch.randn_like(x)
            variance_t = self.extract(self.variance, t, x.shape)
            sigma_t = torch.sqrt(variance_t)
            return mean + sigma_t * noise
        else:
            return mean

    @torch.no_grad()
    def sample(self, model, n=16, channels=1, image_size=28, clip_denoised=True, device="cpu"):
        model.eval()
        x = torch.randn((n, channels, image_size, image_size), device=device)
        
        for i in reversed(range(self.num_timesteps)):
            t = torch.full((n,), i, device=device, dtype=torch.long)
            x = self.p_sample(model, x, t, i, clip_denoised=clip_denoised)
                
        model.train()
        return torch.clamp(x, -1.0, 1.0)

    @torch.no_grad()
    def sample_ddim(self, model, n=16, channels=1, image_size=28, ddim_timesteps=50, eta=0.0, clip_denoised=True, device="cpu"):
        model.eval()
        step_size = self.num_timesteps // ddim_timesteps
        time_steps = list(reversed(range(0, self.num_timesteps, step_size)))
        
        x = torch.randn((n, channels, image_size, image_size), device=device)
        
        for i, t_val in enumerate(time_steps):
            t = torch.full((n,), t_val, device=device, dtype=torch.long)
            t_prev_val = time_steps[i + 1] if i < len(time_steps) - 1 else 0
            
            alpha_bar_t = self.extract(self.alphas_cumprod, t, x.shape)
            t_prev_tensor = torch.full((n,), t_prev_val, device=device, dtype=torch.long)
            alpha_bar_t_prev = self.extract(self.alphas_cumprod, t_prev_tensor, x.shape)
            
            pred_noise = model(x, t)
            
            pred_x0 = (x - torch.sqrt(1.0 - alpha_bar_t) * pred_noise) / torch.sqrt(alpha_bar_t)
            if clip_denoised:
                pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)
            
            sigma_t = eta * torch.sqrt(
                (1 - alpha_bar_t_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_t_prev)
            )
            
            dir_xt = torch.sqrt(1.0 - alpha_bar_t_prev - sigma_t**2) * pred_noise
            noise = torch.randn_like(x) if t_prev_val > 0 else 0.0
            x = torch.sqrt(alpha_bar_t_prev) * pred_x0 + dir_xt + sigma_t * noise
                
        model.train()
        return torch.clamp(x, -1.0, 1.0)

## 4. Inisialisasi & Persiapan Training
Blok pengujian awal untuk memastikan arsitektur berjalan lancar. Tim training dapat menggunakan *instance* `model` dan metode `diffusion.p_losses()` dari blok ini ke dalam *training loop*.

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Status Device: {device}")

# Inisialisasi U-Net dan Gaussian Diffusion
model = UNet(in_channels=1, base_channels=64, time_emb_dim=256).to(device)
diffusion = GaussianDiffusion(num_timesteps=1000).to(device)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Total Parameter U-Net: {count_parameters(model):,}")

# --- Sanity Check Cepat untuk Tim Training ---
dummy_x0 = torch.randn(8, 1, 28, 28, device=device) # Dummy batch size 8
dummy_t = torch.randint(0, 1000, (8,), device=device, dtype=torch.long)

# Menguji kalkulasi loss (Forward process + U-Net prediction otomatis)
loss = diffusion.p_losses(model, dummy_x0, dummy_t)
print(f"Nilai Loss Awal (sebelum training): {loss.item():.4f}")
print("\nArsitektur siap digunakan! Silakan lanjutkan dengan DataLoader dan Optimizer.")

Status Device: cpu
Total Parameter U-Net: 3,399,041
Nilai Loss Awal (sebelum training): 1.0161

Arsitektur siap digunakan! Silakan lanjutkan dengan DataLoader dan Optimizer.
